# Imports & Functions

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [2]:
def clean_csv(path: str) -> pd.DataFrame:
    with open(path, "r") as f:
        lines = [line.strip().split(",") for line in f]
    n_cols = max(len(line) for line in lines)
    normalized = [line + [""] * (n_cols - len(line)) if len(line) < n_cols else line[:n_cols] for line in lines]
    df = pd.DataFrame(normalized)
    df.replace('', np.nan, inplace=True)
    index_values = list(range(50, 1050, 50)) + list(range(2000, 10001, 2000))
    df.index = index_values
    df.index.name = "nbSim"
    return df

def compute_references(df: pd.DataFrame) -> pd.DataFrame:
    df_res = df.copy()
    df_res["mean"] = df.astype(float).mean(axis=1, skipna=True)
    df_res["var"] = df.astype(float).var(axis=1, skipna=True)
    df_res["var_normalized"] = df_res["var"] / df_res.index
    df_res["ci_margin"] = (df_res["var_normalized"] * 1.96)
    df_res["ci_lower"] = df_res["mean"] - df_res["ci_margin"]
    df_res["ci_upper"] = df_res["mean"] + df_res["ci_margin"]
    return df_res

def summarize_results(df_dict: dict[str, pd.DataFrame], idx: int) -> pd.DataFrame:
    summary = {}

    for name, df in df_dict.items():
        mean = df.loc[idx, "mean"]
        var = df.loc[idx, "var_normalized"]
        ci_str = f"[{df.loc[idx, 'ci_lower']:.4f} ; {df.loc[idx, 'ci_upper']:.4f}]"

        summary[name] = {
            "mean": mean,
            "var": var,
            "ci_interval": ci_str
        }

    return pd.DataFrame(summary)

# Monte Carlo Basket Underlying Call

In [3]:
call_mc_path = r"Outputs/MonteCarlo_Simulations_Convergence_Call_Basket.csv"
call_mc_cv_path = r"Outputs/MonteCarlo_ControlVariate_Simulations_Convergence_Call_Basket.csv"
call_mc_an_path = r"Outputs/MonteCarlo_Antithetic_Simulations_Convergence_Call_Basket.csv"
call_mc_an_cv_path = r"Outputs/MonteCarlo_Antithetic_ControlVariate_Simulations_Convergence_Call_Basket.csv"

In [4]:
df_call_mc = clean_csv(call_mc_path)
df_call_mc = compute_references(df_call_mc)

df_call_mc_cv = clean_csv(call_mc_cv_path)
df_call_mc_cv = compute_references(df_call_mc_cv)

df_call_mc_an = clean_csv(call_mc_an_path)
df_call_mc_an = compute_references(df_call_mc_an)

df_call_mc_an_cv = clean_csv(call_mc_an_cv_path)
df_call_mc_an_cv = compute_references(df_call_mc_an_cv)

In [5]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_call_mc.index,
    y=df_call_mc["mean"],
    mode='lines',
    name='Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_cv.index,
    y=df_call_mc_cv["mean"],
    mode='lines',
    name='Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_an.index,
    y=df_call_mc_an["mean"],
    mode='lines',
    name='Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_an_cv.index,
    y=df_call_mc_an_cv["mean"],
    mode='lines',
    name='Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Call Price Convergence by method of Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Price",
    legend_title="Simulation methods"
)

fig.show()

In [6]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_call_mc.index,
    y=df_call_mc["var_normalized"],
    mode='lines',
    name='Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_cv.index,
    y=df_call_mc_cv["var_normalized"],
    mode='lines',
    name='Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_an.index,
    y=df_call_mc_an["var_normalized"],
    mode='lines',
    name='Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_an_cv.index,
    y=df_call_mc_an_cv["var_normalized"],
    mode='lines',
    name='Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Call Variance Convergence by method of Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Variance",
    legend_title="Simulation methods"
)

fig.show()

In [7]:
df_dict = {
    "MonteCarlo_Simulations_Call_Basket": df_call_mc,
    "MonteCarlo_ControlVariate_Simulations_Call_Basket": df_call_mc_cv,
    "MonteCarlo_Antithetic_Simulations_Call_Basket": df_call_mc_an,
    "MonteCarlo_Antithetic_ControlVariate_Simulations_Call_Basket": df_call_mc_an_cv
}

df_summarize = summarize_results(df_dict, 750)
df_summarize

,MonteCarlo_Simulations_Call_Basket,MonteCarlo_ControlVariate_Simulations_Call_Basket,MonteCarlo_Antithetic_Simulations_Call_Basket,MonteCarlo_Antithetic_ControlVariate_Simulations_Call_Basket
mean,11.36181,11.025162,11.112208,10.92375
var,0.383006,0.014727,0.50762,0.012899
ci_interval,[10.6111 ; 12.1125],[10.9963 ; 11.0540],[10.1173 ; 12.1071],[10.8985 ; 10.9490]


# Monte Carlo Basket Underlying Put

In [8]:
put_mc_path = r"Outputs/MonteCarlo_Simulations_Convergence_Put_Basket.csv"
put_mc_cv_path = r"Outputs/MonteCarlo_ControlVariate_Simulations_Convergence_Put_Basket.csv"
put_mc_an_path = r"Outputs/MonteCarlo_Antithetic_Simulations_Convergence_Put_Basket.csv"
put_mc_an_cv_path = r"Outputs/MonteCarlo_Antithetic_ControlVariate_Simulations_Convergence_Put_Basket.csv"

In [9]:
df_put_mc = clean_csv(put_mc_path)
df_put_mc = compute_references(df_put_mc)

df_put_mc_cv = clean_csv(put_mc_cv_path)
df_put_mc_cv = compute_references(df_put_mc_cv)

df_put_mc_an = clean_csv(put_mc_an_path)
df_put_mc_an = compute_references(df_put_mc_an)

df_put_mc_an_cv = clean_csv(put_mc_an_cv_path)
df_put_mc_an_cv = compute_references(df_put_mc_an_cv)

In [10]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_put_mc.index,
    y=df_put_mc["mean"],
    mode='lines',
    name='Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_cv.index,
    y=df_put_mc_cv["mean"],
    mode='lines',
    name='Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_an.index,
    y=df_put_mc_an["mean"],
    mode='lines',
    name='Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_an_cv.index,
    y=df_put_mc_an_cv["mean"],
    mode='lines',
    name='Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Put Price Convergence by method of Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Price",
    legend_title="Simulation methods"
)

fig.show()

In [11]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_put_mc.index,
    y=df_put_mc["var_normalized"],
    mode='lines',
    name='Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_cv.index,
    y=df_put_mc_cv["var_normalized"],
    mode='lines',
    name='Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_an.index,
    y=df_put_mc_an["var_normalized"],
    mode='lines',
    name='Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_an_cv.index,
    y=df_put_mc_an_cv["var_normalized"],
    mode='lines',
    name='Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Put Variance Convergence by method of Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Variance",
    legend_title="Simulation methods"
)

fig.show()

In [12]:
df_dict_put = {
    "MonteCarlo_Simulations_Put_Basket": df_put_mc,
    "MonteCarlo_ControlVariate_Simulations_Put_Basket": df_put_mc_cv,
    "MonteCarlo_Antithetic_Simulations_Put_Basket": df_put_mc_an,
    "MonteCarlo_Antithetic_ControlVariate_Simulations_Put_Basket": df_put_mc_an_cv
}

df_summarize_put = summarize_results(df_dict_put, 750)
df_summarize_put

,MonteCarlo_Simulations_Put_Basket,MonteCarlo_ControlVariate_Simulations_Put_Basket,MonteCarlo_Antithetic_Simulations_Put_Basket,MonteCarlo_Antithetic_ControlVariate_Simulations_Put_Basket
mean,6.522318,6.128385,5.845686,5.979901
var,0.104099,0.004519,0.102492,0.005319
ci_interval,[6.3183 ; 6.7264],[6.1195 ; 6.1372],[5.6448 ; 6.0466],[5.9695 ; 5.9903]


# Quasi Monte Carlo Basket Underlying Call

In [13]:
call_quasi_mc_path = r"Outputs/Quasi_MonteCarlo_Simulations_Convergence_Call_Basket.csv"
call_quasi_mc_cv_path = r"Outputs/Quasi_MonteCarlo_ControlVariate_Simulations_Convergence_Call_Basket.csv"
call_quasi_mc_an_path = r"Outputs/Quasi_MonteCarlo_Antithetic_Simulations_Convergence_Call_Basket.csv"
call_quasi_mc_an_cv_path = r"Outputs/Quasi_MonteCarlo_Antithetic_ControlVariate_Simulations_Convergence_Call_Basket.csv"

In [14]:
df_call_quasi_mc = clean_csv(call_quasi_mc_path)
df_call_quasi_mc = compute_references(df_call_quasi_mc)

df_call_quasi_mc_cv = clean_csv(call_quasi_mc_cv_path)
df_call_quasi_mc_cv = compute_references(df_call_quasi_mc_cv)

df_call_quasi_mc_an = clean_csv(call_quasi_mc_an_path)
df_call_quasi_mc_an = compute_references(df_call_quasi_mc_an)

df_call_quasi_mc_an_cv = clean_csv(call_quasi_mc_an_cv_path)
df_call_quasi_mc_an_cv = compute_references(df_call_quasi_mc_an_cv)

In [15]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_call_quasi_mc.index,
    y=df_call_quasi_mc["mean"],
    mode='lines',
    name='Quasi Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_call_quasi_mc_cv.index,
    y=df_call_quasi_mc_cv["mean"],
    mode='lines',
    name='Quasi Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_call_quasi_mc_an.index,
    y=df_call_quasi_mc_an["mean"],
    mode='lines',
    name='Quasi Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_call_quasi_mc_an_cv.index,
    y=df_call_quasi_mc_an_cv["mean"],
    mode='lines',
    name='Quasi Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Call Price Convergence by method of Quasi Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Price",
    legend_title="Simulation methods"
)

fig.show()

In [16]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_call_quasi_mc.index,
    y=df_call_quasi_mc["var_normalized"],
    mode='lines',
    name='Quasi Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_call_quasi_mc_cv.index,
    y=df_call_quasi_mc_cv["var_normalized"],
    mode='lines',
    name='Quasi Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_call_quasi_mc_an.index,
    y=df_call_quasi_mc_an["var_normalized"],
    mode='lines',
    name='Quasi Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_call_quasi_mc_an_cv.index,
    y=df_call_quasi_mc_an_cv["var_normalized"],
    mode='lines',
    name='Quasi Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Call Variance Convergence by method of Quasi Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Variance",
    legend_title="Simulation methods"
)

fig.show()

In [17]:
df_dict_quasi = {
    "Quasi_MonteCarlo_Simulations_Call_Basket": df_call_quasi_mc,
    "Quasi_MonteCarlo_ControlVariate_Simulations_Call_Basket": df_call_quasi_mc_cv,
    "Quasi_MonteCarlo_Antithetic_Simulations_Call_Basket": df_call_quasi_mc_an,
    "Quasi_MonteCarlo_Antithetic_ControlVariate_Simulations_Call_Basket": df_call_quasi_mc_an_cv
}

df_summarize_quasi = summarize_results(df_dict_quasi, 750)
df_summarize_quasi

,Quasi_MonteCarlo_Simulations_Call_Basket,Quasi_MonteCarlo_ControlVariate_Simulations_Call_Basket,Quasi_MonteCarlo_Antithetic_Simulations_Call_Basket,Quasi_MonteCarlo_Antithetic_ControlVariate_Simulations_Call_Basket
mean,0.822927,9.679401,0.852096,9.676889
var,0.001395,0.00001,0.001436,0.00001
ci_interval,[0.8202 ; 0.8257],[9.6794 ; 9.6794],[0.8493 ; 0.8549],[9.6769 ; 9.6769]


# Quasi Monte Carlo Basket Underlying Put

In [18]:
put_quasi_mc_path = r"Outputs/Quasi_MonteCarlo_Simulations_Convergence_Put_Basket.csv"
put_quasi_mc_cv_path = r"Outputs/Quasi_MonteCarlo_ControlVariate_Simulations_Convergence_Put_Basket.csv"
put_quasi_mc_an_path = r"Outputs/Quasi_MonteCarlo_Antithetic_Simulations_Convergence_Put_Basket.csv"
put_quasi_mc_an_cv_path = r"Outputs/Quasi_MonteCarlo_Antithetic_ControlVariate_Simulations_Convergence_Put_Basket.csv"

In [19]:
df_put_quasi_mc = clean_csv(put_quasi_mc_path)
df_put_quasi_mc = compute_references(df_put_quasi_mc)

df_put_quasi_mc_cv = clean_csv(put_quasi_mc_cv_path)
df_put_quasi_mc_cv = compute_references(df_put_quasi_mc_cv)

df_put_quasi_mc_an = clean_csv(put_quasi_mc_an_path)
df_put_quasi_mc_an = compute_references(df_put_quasi_mc_an)

df_put_quasi_mc_an_cv = clean_csv(put_quasi_mc_an_cv_path)
df_put_quasi_mc_an_cv = compute_references(df_put_quasi_mc_an_cv)

In [20]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_put_quasi_mc.index,
    y=df_put_quasi_mc["mean"],
    mode='lines',
    name='Quasi Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_put_quasi_mc_cv.index,
    y=df_put_quasi_mc_cv["mean"],
    mode='lines',
    name='Quasi Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_put_quasi_mc_an.index,
    y=df_put_quasi_mc_an["mean"],
    mode='lines',
    name='Quasi Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_put_quasi_mc_an_cv.index,
    y=df_put_quasi_mc_an_cv["mean"],
    mode='lines',
    name='Quasi Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Put Price Convergence by method of Quasi Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Price",
    legend_title="Simulation methods"
)

fig.show()

In [21]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_put_quasi_mc.index,
    y=df_put_quasi_mc["var_normalized"],
    mode='lines',
    name='Quasi Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_put_quasi_mc_cv.index,
    y=df_put_quasi_mc_cv["var_normalized"],
    mode='lines',
    name='Quasi Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_put_quasi_mc_an.index,
    y=df_put_quasi_mc_an["var_normalized"],
    mode='lines',
    name='Quasi Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_put_quasi_mc_an_cv.index,
    y=df_put_quasi_mc_an_cv["var_normalized"],
    mode='lines',
    name='Quasi Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Put Variance Convergence by method of Quasi Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Variance",
    legend_title="Simulation methods"
)

fig.show()

In [22]:
df_dict_put_quasi = {
    "Quasi_MonteCarlo_Simulations_Put_Basket": df_put_quasi_mc,
    "Quasi_MonteCarlo_ControlVariate_Simulations_Put_Basket": df_put_quasi_mc_cv,
    "Quasi_MonteCarlo_Antithetic_Simulations_Put_Basket": df_put_quasi_mc_an,
    "Quasi_MonteCarlo_Antithetic_ControlVariate_Simulations_Put_Basket": df_put_quasi_mc_an_cv
}

df_summarize_put = summarize_results(df_dict_put_quasi, 750)
df_summarize_put

,Quasi_MonteCarlo_Simulations_Put_Basket,Quasi_MonteCarlo_ControlVariate_Simulations_Put_Basket,Quasi_MonteCarlo_Antithetic_Simulations_Put_Basket,Quasi_MonteCarlo_Antithetic_ControlVariate_Simulations_Put_Basket
mean,0.440057,7.059397,0.470782,7.060539
var,0.000751,0.000027,0.000825,0.000027
ci_interval,[0.4386 ; 0.4415],[7.0593 ; 7.0595],[0.4692 ; 0.4724],[7.0605 ; 7.0606]
